# MuonClip angular-residual WeightWatcher analysis

This notebook tests a very specific hypothesis:

**MuonClip may leave a large, nearly random radial singular-value sector in each trained weight matrix, while the learned structure is carried mainly by changes in the singular-vector geometry.**

For each transformer matrix we write

$$
W_t = U_t \Sigma_t V_t^\top .
$$

The radial sector is the singular-value matrix

$$
\Sigma_t .
$$

The polar/angular factor is

$$
Q_t = U_t V_t^\top .
$$

If we analyze \(Q_t\) directly with WeightWatcher, the spectrum is trivial because all nonzero singular values of \(Q_t\) are one. Therefore the correct matrix for ordinary heavy-tailed spectral analysis is **not** \(Q_t\) itself.

Instead we isolate the learned angular displacement relative to initialization:

$$
A_t = Q_t - Q_0 .
$$

This removes the radial singular-value sector at both endpoints and leaves a matrix whose singular values measure how far the learned angular map has moved away from the initialization. For square matrices, if the relative rotation has eigenangles \(\theta_j\), then the squared singular values of \(A_t\) are directly related to the angular motion.

To preserve the natural layer units without changing any power-law exponent, we multiply the angular residual by one scalar per layer,

$$
\widetilde A_t = \bar{\sigma}_0 (Q_t-Q_0),
$$

where \(\bar{\sigma}_0\) is the mean initial singular value. Multiplying a matrix by a scalar rescales eigenvalues but does not change the heavy-tail exponent.

The notebook then **puts these angular-residual matrices back into a copy of the GPT model** and runs the standard WeightWatcher analysis on the six transformer matrices.

We compare:

1. the raw initialization;
2. the raw trained/final model;
3. the pure polar factors \(Q_0\) and \(Q_t\) as a sanity check;
4. the angular-residual model \(\widetilde A_t\), which is the main object of interest.

The key question is whether WeightWatcher sees a nontrivial HTSR spectrum in the angular residual even when the raw trained matrix remains close to a random/MP-like bulk.


## Interpretation

The raw WeightWatcher ESD answers:

$$
\text{What is the singular-value spectrum of }W_t?
$$

The angular-residual ESD answers:

$$
\text{What is the singular-value spectrum of the change in the polar/angular factor?}
$$

This is an exact removal of the instantaneous radial sector. It is **not** SVDSharpness, entry shuffling, MP clipping, or generic denoising.

A useful outcome would be:

$$
\alpha_{\mathrm{raw}}
\text{ remains large/random-like}
$$

while

$$
\alpha_{\mathrm{angular\ residual}}
$$

moves into a heavy-tailed regime with a good fit and substantial RAND distance.

A negative result is equally informative: if the angular-residual ESD is not heavy-tailed, then the learned angular structure is not described by the ordinary HTSR singular-value theory in this representation.


In [ ]:
import os

TARGET_SEED = int(os.environ.get("TARGET_SEED", "4242"))
TARGET_OPTIMIZER = os.environ.get("TARGET_OPTIMIZER", "muon_clip")
RUN_DIR = os.environ.get("RUN_DIR", "")
OUTPUT_ROOT = os.environ.get("ANGULAR_RESIDUAL_OUTPUT", "")
WW_MIN_EVALS = int(os.environ.get("WW_MIN_EVALS", "20"))


In [ ]:
from pathlib import Path
import copy
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display, Image

os.environ.setdefault("MPLBACKEND", "Agg")

# Find the experiment root.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
EXPERIMENT_ROOT = next(
    p for p in candidates
    if (p / "src" / "rg_nanogpt_one_head" / "model.py").is_file()
)
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from rg_nanogpt_one_head.angular_weightwatcher_core import (
    AnalysisConfig,
    resolve_run,
    load_weight_pairs,
    _load_payload,
    _model_config,
    _build_model,
    polar,
)
from rg_nanogpt_one_head.model import transformer_matrix_items
from rg_nanogpt_one_head.spectral import WeightMatrixHolder, _attach_matrix_metadata

import weightwatcher as ww

config = AnalysisConfig(
    seed=TARGET_SEED,
    optimizer=TARGET_OPTIMIZER,
    run_dir=RUN_DIR or None,
    show_plots=True,
)

resolved = resolve_run(config)
initial_weights, final_weights, metadata = load_weight_pairs(config, resolved)

initial_payload = _load_payload(resolved.initial_path)
final_payload = _load_payload(resolved.final_path)
model_cfg = _model_config(initial_payload, final_payload, resolved.run_dir)
model_initial = _build_model(initial_payload, model_cfg)
model_final = _build_model(final_payload, model_cfg)

OUTPUT_DIR = (
    Path(OUTPUT_ROOT).expanduser().resolve()
    if OUTPUT_ROOT
    else resolved.run_dir / "diagnostics" / "muonclip_angular_residual_weightwatcher"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_DIR       =", resolved.run_dir)
print("INITIAL       =", resolved.initial_path)
print("FINAL         =", resolved.final_path)
print("OUTPUT_DIR    =", OUTPUT_DIR)
print("MATRICES      =", list(initial_weights))


## Construct the Muon-radial-removed matrices

For every layer:

1. compute the compact SVD of the initial and final matrices;
2. discard all singular values;
3. reconstruct the polar factors \(Q_0\) and \(Q_t\);
4. form the angular displacement \(Q_t-Q_0\);
5. multiply by the mean initial singular value only to retain the layer's natural numerical scale.

The scale multiplication is irrelevant to the fitted exponent but makes the transformed model numerically easier to compare with the original model.


In [ ]:
def angular_residual(initial: np.ndarray, final: np.ndarray):
    initial = np.asarray(initial, dtype=np.float64)
    final = np.asarray(final, dtype=np.float64)

    q0 = polar(initial)
    qt = polar(final)

    s0 = np.linalg.svd(initial, compute_uv=False)
    scale = float(np.mean(s0))

    residual = scale * (qt - q0)
    return q0, qt, residual, scale


angular_data = {}
rows = []

for name in initial_weights:
    w0 = initial_weights[name]
    wt = final_weights[name]
    q0, qt, residual, scale = angular_residual(w0, wt)

    angular_data[name] = {
        "W0": w0,
        "Wt": wt,
        "Q0": q0,
        "Qt": qt,
        "A": residual,
        "scale": scale,
    }

    rows.append({
        "matrix_name": name,
        "shape": str(w0.shape),
        "mean_sigma_initial": scale,
        "raw_relative_change": np.linalg.norm(wt-w0) / np.linalg.norm(w0),
        "polar_relative_change": np.linalg.norm(qt-q0) / np.linalg.norm(q0),
        "angular_residual_norm": np.linalg.norm(residual),
    })

diagnostics = pd.DataFrame(rows)
display(diagnostics)


## Put the transformed matrices back into an actual GPT model

We now make two model copies:

- `model_polar_final`: each transformer matrix is replaced by its final polar factor \(Q_t\);
- `model_angular_residual`: each transformer matrix is replaced by the scaled angular residual \(\widetilde A_t\).

The embedding, layer norms, and all non-transformer parameters are left untouched.

WeightWatcher is then run through the same `WeightMatrixHolder` used by the baseline code, so the analyzed inventory is exactly the six matrices:

- `L00_W_Q`
- `L00_W_K`
- `L00_W_V`
- `L00_W_O`
- `L00_W_MLP_IN`
- `L00_W_MLP_OUT`


In [ ]:
def replace_transformer_matrices(model, matrices):
    model = copy.deepcopy(model).cpu().eval()
    with torch.no_grad():
        for name, _, _, parameter in transformer_matrix_items(model):
            target = torch.as_tensor(
                matrices[name],
                dtype=parameter.dtype,
                device=parameter.device,
            )
            if tuple(target.shape) != tuple(parameter.shape):
                raise ValueError(
                    f"shape mismatch for {name}: {target.shape} vs {parameter.shape}"
                )
            parameter.copy_(target)
    return model


polar_final_matrices = {
    name: angular_data[name]["Qt"] for name in angular_data
}
angular_residual_matrices = {
    name: angular_data[name]["A"] for name in angular_data
}

model_polar_final = replace_transformer_matrices(
    model_final, polar_final_matrices
)
model_angular_residual = replace_transformer_matrices(
    model_final, angular_residual_matrices
)

print("Transformed GPT copies created.")


## Run native WeightWatcher

The same WeightWatcher call is used for all four model states.

We request:

- `plot=True`
- `savefig=<directory>`
- `randomize=True`
- `ERG=False`

so we obtain the original WeightWatcher log-log ESD plots, the fitted heavy-tail exponent \(\alpha\), KS statistic \(D\), and `rand_distance`.

The polar-factor model is included only as a sanity check. Because its nonzero singular values are all one, a meaningful HTSR fit is not expected there.

The **angular-residual model is the primary test**.


In [ ]:
def run_ww(label, model):
    savedir = OUTPUT_DIR / label
    savedir.mkdir(parents=True, exist_ok=True)

    holder = WeightMatrixHolder(model)
    watcher = ww.WeightWatcher(model=holder)

    details = watcher.analyze(
        plot=True,
        savefig=str(savedir),
        min_evals=WW_MIN_EVALS,
        randomize=True,
        ERG=False,
    )

    frame = _attach_matrix_metadata(
        pd.DataFrame(details),
        holder.matrix_metadata,
    )
    frame.insert(0, "state", label)

    keep = [
        c for c in [
            "state", "matrix_name", "layer_id", "alpha", "D",
            "rand_distance", "xmin", "xmax", "num_evals",
            "stable_rank", "entropy"
        ]
        if c in frame.columns
    ]

    print(f"\n===== {label} =====")
    display(frame[keep])

    # Always display every WeightWatcher PNG produced for this state.
    pngs = sorted(savedir.rglob("*.png"))
    if not pngs:
        print("WARNING: WeightWatcher produced no PNG files in", savedir)
    for path in pngs:
        print(path.name)
        display(Image(filename=str(path)))

    return frame


frames = {}
frames["raw_initial"] = run_ww("raw_initial", model_initial)
frames["raw_final"] = run_ww("raw_final", model_final)
frames["polar_final"] = run_ww("polar_final", model_polar_final)
frames["angular_residual"] = run_ww(
    "angular_residual",
    model_angular_residual,
)


## Direct alpha comparison

The table below places the raw and radial-removed analyses side-by-side.

The most important comparison is:

$$
\alpha_{\mathrm{raw\ final}}
\quad\text{versus}\quad
\alpha_{\mathrm{angular\ residual}}.
$$

Do not interpret `polar_final` alpha as a physical HTSR exponent: the polar factor has a deliberately flat singular spectrum.


In [ ]:
all_results = pd.concat(frames.values(), ignore_index=True)

cols = [
    c for c in [
        "state", "matrix_name", "alpha", "D", "rand_distance",
        "xmin", "xmax", "num_evals"
    ]
    if c in all_results.columns
]
summary = all_results[cols].sort_values(["matrix_name", "state"])
display(summary)

pivot_alpha = summary.pivot(
    index="matrix_name",
    columns="state",
    values="alpha",
)
print("\nALPHA COMPARISON")
display(pivot_alpha)

summary_path = OUTPUT_DIR / "weightwatcher_angular_residual_summary.csv"
summary.to_csv(summary_path, index=False)
print("Saved:", summary_path)


## Sanity checks

A valid run should satisfy:

1. The initial and final raw models reproduce the familiar WeightWatcher spectra.
2. The polar-final spectrum is concentrated at one, confirming that the radial singular values were actually removed.
3. The angular-residual spectrum is nontrivial whenever the learned polar map differs from initialization.
4. Any heavy-tail interpretation should be based on the angular-residual fit quality (`D`, fitted tail extent if reported, and `rand_distance`), not on alpha alone.

If the angular-residual matrices show substantially stronger HTSR structure than the raw MuonClip matrices, that supports the hypothesis that the Muon radial sector was masking learned angular correlations.

If they do not, then the angular learning may require a different observable than ordinary singular-value HTSR.
